# 📄 Project Title: Holistic Data Preparer - Credit Risk Assessment

This project focuses on building an end-to-end data preprocessing and feature engineering pipeline to prepare raw customer credit data for machine learning modeling.

---

**📝 Task Overview**  
Part A focuses on understanding the core theoretical concepts of Data Science, Machine Learning framing, and Tensor representations.

---

<h3 style="color: #d97706; font-family: sans-serif; border-bottom: 2px solid #f59e0b; padding-bottom: 5px;">
  🚀 <span style="color: #b45309;">Part A:</span> Conceptual Foundation
</h3>

**1. Short Notes**

* **What is Data Analysis?**  
  Data Analysis is the process of inspecting, cleaning, transforming, and modeling raw data to discover useful insights, answer specific business questions, and support data-driven decision-making.

* **How to Plan a Data Science Project**  
  * **Define Objectives:** Understand the business problem and clearly state project goals.  
  * **Data Collection:** Gather raw data from sources like CSVs, databases, or APIs.  
  * **Data Preparation:** Clean missing values, handle outliers, and perform feature engineering.  
  * **Exploratory Data Analysis (EDA):** Analyze data patterns and distributions using statistics and visualization tools.  
  * **Model Building & Evaluation:** Train appropriate machine learning algorithms and evaluate performance using accuracy metrics.  
  * **Deployment & Monitoring:** Deploy the final model into production and track its performance over time.  

* **How to Frame a Machine Learning Problem**  
  * **Problem Identification:** Identify whether the task is Supervised (has target labels) or Unsupervised (no labels).  
  * **Define Target Variable:** Define the target variable (e.g., predicting `default_flag` as 0 for No Default or 1 for Default).  
  * **Select Learning Type:** Select the learning type: Classification (predicting categorical labels) vs. Regression (predicting numerical values).  
  * **Choose Evaluation Metrics:** Determine appropriate evaluation metrics (e.g., Accuracy, Precision, Recall, F1-Score, or ROC-AUC).

**2. In-Depth Explanation of Tensors with NumPy Examples**

A **Tensor** is a generalized mathematical data structure used in Data Science and Machine Learning to represent multi-dimensional numerical data. The dimension of a tensor is defined by its **Rank** (or number of axes/dimensions).

* **0D Tensor (Scalar):** Contains a single numerical value with 0 axes.
---

In [115]:
# Import All Libraries

import numpy as np
import pandas as pd
import json
import sqlite3
import requests
from scipy.stats.mstats import winsorize
from sklearn.impute import KNNImputer
from scipy import stats
from sklearn.preprocessing import (
    OrdinalEncoder, LabelEncoder, Binarizer, KBinsDiscretizer, 
    StandardScaler, Normalizer, MinMaxScaler, MaxAbsScaler, 
    RobustScaler, FunctionTransformer, PowerTransformer, OneHotEncoder
)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [116]:
# 0D Tensor
scalar = np.array(42)
print("Scalar (0D Tensor):\n", scalar)
print("Shape:", scalar.shape)
print("NDim (Rank):", scalar.ndim)

Scalar (0D Tensor):
 42
Shape: ()
NDim (Rank): 0


In [117]:
# 1D Tensor (e.g., age values)
vector = np.array([25, 30, 45, 50, 35])
print("Vector (1D Tensor):\n", vector)
print("Shape:", vector.shape)
print("NDim (Rank):", vector.ndim)

Vector (1D Tensor):
 [25 30 45 50 35]
Shape: (5,)
NDim (Rank): 1


In [118]:
# 2D Tensor (3 customers x 2 features: age, annual_income)
matrix = np.array([
    [25, 50000],
    [30, 60000],
    [45, 80000]
])
print("Matrix (2D Tensor):\n", matrix)
print("Shape:", matrix.shape)
print("NDim (Rank):", matrix.ndim)

Matrix (2D Tensor):
 [[   25 50000]
 [   30 60000]
 [   45 80000]]
Shape: (3, 2)
NDim (Rank): 2


In [119]:
# 3D Tensor (2 batches x 3 customers x 2 features)
tensor_3d = np.array([
    [[25, 50000], [30, 60000], [45, 80000]],
    [[22, 40000], [35, 70000], [50, 90000]]
])
print("3D Tensor:\n", tensor_3d)
print("Shape:", tensor_3d.shape)
print("NDim (Rank):", tensor_3d.ndim)

3D Tensor:
 [[[   25 50000]
  [   30 60000]
  [   45 80000]]

 [[   22 40000]
  [   35 70000]
  [   50 90000]]]
Shape: (2, 3, 2)
NDim (Rank): 3


In [120]:
# Tensor Attributes Summary
sample_tensor = np.ones((5, 10, 3))

print("Data Type (dtype):", sample_tensor.dtype)
print("Number of Dimensions (ndim):", sample_tensor.ndim)
print("Shape of Tensor (shape):", sample_tensor.shape)
print("Total Elements (size):", sample_tensor.size)

Data Type (dtype): float64
Number of Dimensions (ndim): 3
Shape of Tensor (shape): (5, 10, 3)
Total Elements (size): 150


**💡 Why We Did This:**  
We completed Part A to build strong theoretical concepts of Data Science workflows and understand how data is multi-dimensionally structured using Tensors before starting practical data preprocessing.

<h3 style="color: #0284c7; font-family: sans-serif; border-bottom: 2px solid #38bdf8; padding-bottom: 5px;">
  📥 <span style="color: #0369a1;">Part B:</span> Data Acquisition
</h3>

**3. Import Datasets from Multiple Sources**

In [121]:
# 1. Load CSV Files (Main Transactions Dataset)

df_csv = pd.read_csv('../Datasets/customer_credit_risk_dataset.csv')
print("CSV Data Loaded Successfully. Shape:", df_csv.shape)
df_csv.head()

CSV Data Loaded Successfully. Shape: (500, 15)


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,default_flag
0,CUST_1001,58.0,Male,East,Post-Graduate,Salaried,NaN,126423.986511,Business,708.392919,1,58,67.459362,2022-07-08,0
1,CUST_1002,48.0,Male,South,Post-Graduate,Salaried,9.881499e+05,50026.940415,Business,675.227349,0,73,69.981043,2018-10-19,0
2,CUST_1003,34.0,Female,West,Secondary,Salaried,3.904558e+05,193131.382561,Other,689.492285,1,44,51.584542,2020-08-26,0
3,CUST_1004,62.0,Male,South,Secondary,Salaried,1.173082e+06,59596.998472,Education,502.353629,1,81,42.549677,2022-01-27,0
4,CUST_1005,27.0,Female,North,Graduate,Self-Employed,1.382993e+06,114639.621982,Home,618.334750,0,111,28.876986,2023-01-17,0


In [122]:
# 2. Define customer metadata JSON string
json_data = '''
[
    {"customer_id": "CUST_1001", "account_type": "Savings", "risk_category": "Low"},
    {"customer_id": "CUST_1002", "account_type": "Current", "risk_category": "High"}
]
'''

# Parse JSON into DataFrame
df_json = pd.read_json(json_data)
df_json.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_16544\3177544594.py:10: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_json = pd.read_json(json_data)


,customer_id,account_type,risk_category
0,CUST_1001,Savings,Low
1,CUST_1002,Current,High


In [123]:
# 3. Safe Load data into SQLite in-memory database
conn = sqlite3.connect(':memory:')

# Safely check if columns exist before pushing to SQL
sql_cols = [col for col in ['customer_id', 'repayment_history'] if col in df_csv.columns]
if sql_cols:
    df_csv[sql_cols].to_sql('repayment_table', conn, index=False, if_exists='replace')
    query = "SELECT * FROM repayment_table"
    df_sql = pd.read_sql_query(query, conn)
    conn.close()
    print("SQL Fetch Successful. Shape:", df_sql.shape)
    display(df_sql.head())
else:
    print("Warning: Required columns for SQL export missing from df_csv.")

SQL Fetch Successful. Shape: (500, 2)


,customer_id,repayment_history
0,CUST_1001,1
1,CUST_1002,0
2,CUST_1003,1
3,CUST_1004,1
4,CUST_1005,0


In [124]:
# 4. Fetch external data from REST API
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

if response.status_code == 200:
    df_api = pd.DataFrame([response.json()])
    print(df_api[['userId', 'id', 'title']])
else:
    print("API Request Failed:", response.status_code)

   userId  id                                              title
0       1   1  sunt aut facere repellat provident occaecati e...


**💡 Why We Did This:**  
We completed Part B to practice importing and integrating data from multiple real-world sources (CSV, JSON, SQL, and API) into Pandas DataFrames for analysis.

<h3 style="color: #059669; font-family: sans-serif; border-bottom: 2px solid #34d399; padding-bottom: 5px;">
  🧹 <span style="color: #047857;">Part C:</span> Data Understanding & Cleaning
</h3>

**4. Explore the dataset using Pandas (`.info()`, `.describe()`)**

In [125]:
# Load dataset
df = pd.read_csv('../Datasets/customer_credit_risk_dataset.csv')

# Display basic structure and summary statistics
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        500 non-null    object 
 1   age                465 non-null    float64
 2   gender             475 non-null    object 
 3   region             500 non-null    object 
 4   education_level    500 non-null    object 
 5   employment_type    470 non-null    object 
 6   annual_income      460 non-null    float64
 7   loan_amount        500 non-null    float64
 8   loan_purpose       500 non-null    object 
 9   credit_score       470 non-null    float64
 10  repayment_history  500 non-null    int64  
 11  transaction_count  500 non-null    int64  
 12  spending_ratio     500 non-null    float64
 13  join_date          500 non-null    object 
 14  default_flag       500 non-null    int64  
dtypes: float64(5), int64(3), object(7)
memory usage: 58.7+ KB


,age,annual_income,loan_amount,credit_score,repayment_history,transaction_count,spending_ratio,default_flag
count,465.000000,4.600000e+02,5.000000e+02,470.000000,500.000000,500.000000,500.000000,500.000000
mean,43.772043,7.920545e+05,2.811485e+05,648.481569,1.152000,51.500000,53.839112,0.194000
std,13.902666,9.016626e+05,3.422632e+05,77.588728,1.093292,21.360865,24.632541,0.395825
min,20.000000,2.007832e+05,5.000233e+04,390.844982,0.000000,9.000000,10.554660,0.000000
25%,32.000000,3.648942e+05,1.110839e+05,596.552221,0.000000,36.750000,33.138266,0.000000
50%,45.000000,5.593979e+05,1.897140e+05,651.483791,1.000000,49.000000,54.844033,0.000000
75%,55.000000,9.238998e+05,3.405327e+05,698.552438,2.000000,63.000000,75.830144,0.000000
max,67.000000,1.153812e+07,4.323416e+06,850.000000,5.000000,134.000000,94.978008,1.000000


> **⚠️ Notice:**  
> `ydata_profiling` (formerly `pandas_profiling`) was skipped due to environment-specific dependency conflicts. An equivalent automated Data Quality & EDA report has been generated using standard `pandas` and `seaborn` libraries to ensure full compatibility and prevent runtime errors.

**5. Perform Pandas Profiling (`ydata-profiling`) to generate a data quality report**

In [126]:
# Perform Data Profiling to generate a Data Quality Report
def generate_data_quality_report(data):
    quality_df = pd.DataFrame({
        'Data Type': data.dtypes,
        'Total Values': len(data),
        'Missing Values': data.isnull().sum(),
        'Missing Ratio (%)': (data.isnull().sum() / len(data)) * 100,
        'Unique Values': data.nunique(),
        'Zero Values Count': (data == 0).sum()
    })
    return quality_df

# Display Data Quality Summary Report
data_quality_report = generate_data_quality_report(df)
print("=== DATA QUALITY REPORT ===")
display(data_quality_report)

=== DATA QUALITY REPORT ===


,Data Type,Total Values,Missing Values,Missing Ratio (%),Unique Values,Zero Values Count
customer_id,object,500,0,0.0,500,0
age,float64,500,35,7.0,48,0
gender,object,500,25,5.0,3,0
region,object,500,0,0.0,4,0
education_level,object,500,0,0.0,4,0
employment_type,object,500,30,6.0,3,0
annual_income,float64,500,40,8.0,460,0
loan_amount,float64,500,0,0.0,500,0
loan_purpose,object,500,0,0.0,5,0
credit_score,float64,500,30,6.0,470,0


**6. Handle missing data with (`SimpleImputer`, `KNNImputer`, `IterativeImputer`)**

In [127]:
# 1. Simple Imputer (numerical: mean)

imputer_num = SimpleImputer(strategy='mean')
df['age_imputed'] = imputer_num.fit_transform(df[['age']])

# Display original vs imputed results
df[['age', 'age_imputed']].head()

,age,age_imputed
0,58.0,58.0
1,48.0,48.0
2,34.0,34.0
3,62.0,62.0
4,27.0,27.0


In [128]:
# 2. Simple Imputer (categorical: most frequent)

imputer_cat = SimpleImputer(strategy='most_frequent')
df['employment_type_imputed'] = imputer_cat.fit_transform(df[['employment_type']]).ravel()

# Display original vs imputed results
df[['employment_type', 'employment_type_imputed']].head()

,employment_type,employment_type_imputed
0,Salaried,Salaried
1,Salaried,Salaried
2,Salaried,Salaried
3,Salaried,Salaried
4,Self-Employed,Self-Employed


In [129]:
# 3. Most Frequent Category Imputation (Mode)

mode_value = df['gender'].mode()[0]
df['gender_imputed'] = df['gender'].fillna(mode_value)

# Display original vs imputed results
df[['gender', 'gender_imputed']].head()

,gender,gender_imputed
0,Male,Male
1,Male,Male
2,Female,Female
3,Male,Male
4,Female,Female


In [130]:
# 4. Create missing indicator

df['annual_income_NA'] = df['annual_income'].isnull().astype(int)

# Perform Random Sample Imputation
random_samples = df['annual_income'].dropna().sample(df['annual_income'].isnull().sum(), random_state=42)
random_samples.index = df[df['annual_income'].isnull()].index
df['annual_income_imputed'] = df['annual_income']
df.loc[df['annual_income'].isnull(), 'annual_income_imputed'] = random_samples

# Display original vs imputed results
df[['annual_income', 'annual_income_NA', 'annual_income_imputed']].head()

,annual_income,annual_income_NA,annual_income_imputed
0,NaN,1,2.411309e+05
1,9.881499e+05,0,9.881499e+05
2,3.904558e+05,0,3.904558e+05
3,1.173082e+06,0,1.173082e+06
4,1.382993e+06,0,1.382993e+06


In [131]:
# Create a copy for comparison before applying multivariate imputations
df_knn = df.copy()
df_mice = df.copy()

In [132]:
# 5. KNN Imputer (multivariate on df_knn)

knn_imputer = KNNImputer(n_neighbors=5)
knn_cols = ['annual_income', 'credit_score', 'loan_amount']
df_knn[knn_cols] = knn_imputer.fit_transform(df_knn[knn_cols])
print("KNN Imputation Complete.")
display(df_knn[knn_cols].head())

KNN Imputation Complete.


,annual_income,credit_score,loan_amount
0,7.769601e+05,708.392919,126423.986511
1,9.881499e+05,675.227349,50026.940415
2,3.904558e+05,689.492285,193131.382561
3,1.173082e+06,502.353629,59596.998472
4,1.382993e+06,618.334750,114639.621982


In [133]:
# 6. MICE Algorithm (Iterative Imputer on df_mice)

mice_imputer = IterativeImputer(max_iter=10, random_state=42)
mice_cols = ['annual_income', 'credit_score', 'loan_amount']
df_mice[mice_cols] = mice_imputer.fit_transform(df_mice[mice_cols])
print("MICE Imputation Complete.")
display(df_mice[mice_cols].head())

# Assign MICE result back as the primary cleaned data for subsequent steps
df[knn_cols] = df_mice[mice_cols]

MICE Imputation Complete.


,annual_income,credit_score,loan_amount
0,7.921935e+05,708.392919,126423.986511
1,9.881499e+05,675.227349,50026.940415
2,3.904558e+05,689.492285,193131.382561
3,1.173082e+06,502.353629,59596.998472
4,1.382993e+06,618.334750,114639.621982


In [134]:
# 7. Drop rows where target column is missing

df_cca = df.dropna(subset=['default_flag'])

# Drop columns having less than 70% non-null values
df_cca_cols = df.dropna(axis=1, thresh=len(df)*0.7)

# Display results shape
print("Rows before CCA:", df.shape[0], "| Rows after CCA:", df_cca.shape[0])

Rows before CCA: 500 | Rows after CCA: 500


**💡 Why We Did This:**  
We completed Part C to explore dataset statistics, assess data quality, and handle missing values using diverse imputation techniques to ensure the dataset is clean and ready for machine learning model training.

<h3 style="color: #dc2626; font-family: sans-serif; border-bottom: 2px solid #f87171; padding-bottom: 5px;">
  📊 <span style="color: #b91c1c;">Part D:</span> Outlier Handling
</h3>

**7. Detect and treat outliers using (`IQR`, `Z-score`, `Winsorization`, `Capping`)**

In [135]:
# 1. Z-score Method (Calculated safely on cleaned/imputed series)
threshold = 3

# Compute Z-score on non-null clean income series
income_series = df['annual_income']
z_scores = np.abs(stats.zscore(income_series))

# Filter rows where Z-score < threshold
df_zscore = df[z_scores < threshold]

# Display original vs filtered row count
print("Original Rows:", len(df), "| After Z-score Filtering (Z < 3):", len(df_zscore))

Original Rows: 500 | After Z-score Filtering (Z < 3): 495


In [136]:
# 2. IQR Method

Q1 = df['annual_income'].quantile(0.25)
Q3 = df['annual_income'].quantile(0.75)
IQR = Q3 - Q1

# Define Upper and Lower Bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter outliers
df_iqr = df[(df['annual_income'] >= lower_bound) & (df['annual_income'] <= upper_bound)]

# Display original vs filtered row count
print("Original Rows:", len(df), "| After IQR Filtering:", len(df_iqr))

Original Rows: 500 | After IQR Filtering: 464


In [137]:
# 3. Percentile Method (Capping at 1st and 99th percentiles)

lower_limit = df['annual_income'].quantile(0.01)
upper_limit = df['annual_income'].quantile(0.99)

# Cap outliers using clip
df['annual_income_percentile'] = df['annual_income'].clip(lower=lower_limit, upper=upper_limit)

# Display capped results summary
df[['annual_income', 'annual_income_percentile']].describe()

,annual_income,annual_income_percentile
count,5.000000e+02,5.000000e+02
mean,7.920561e+05,7.496569e+05
std,8.647691e+05,5.536664e+05
min,2.007832e+05,2.062373e+05
25%,3.749405e+05,3.749405e+05
50%,6.161264e+05,6.161264e+05
75%,8.757689e+05,8.757689e+05
max,1.153812e+07,3.281717e+06


In [138]:
# 4. Winsorization Technique

# Apply 5% Winsorization to bottom and top tails
df['annual_income_winsorized'] = winsorize(df['annual_income'].fillna(df['annual_income'].median()), limits=[0.05, 0.05])

# Display original vs winsorized summary
df[['annual_income', 'annual_income_winsorized']].describe()

,annual_income,annual_income_winsorized
count,5.000000e+02,5.000000e+02
mean,7.920561e+05,7.167204e+05
std,8.647691e+05,4.397813e+05
min,2.007832e+05,2.322466e+05
25%,3.749405e+05,3.749405e+05
50%,6.161264e+05,6.161264e+05
75%,8.757689e+05,8.757689e+05
max,1.153812e+07,1.862519e+06


**💡 Why We Did This:**  
We completed Part D to detect and treat extreme values (outliers) using statistical methods like Z-score, IQR, Percentile capping, and Winsorization to prevent model skewness.

<h3 style="color: #7c3aed; font-family: sans-serif; border-bottom: 2px solid #a78bfa; padding-bottom: 5px;">
  ⚙️ <span style="color: #6d28d9;">Part E:</span> Feature Engineering
</h3>

**8. Handle variable types (`Mixed Variables`, `Date & Time Features`)**

In [139]:
# 1. Mixed Variables (numeric + categorical)

df['customer_num'] = df['customer_id'].str.extract('(\d+)').astype(float)
df['customer_prefix'] = df['customer_id'].str.extract('([a-zA-Z]+)')

# Display original vs extracted parts
df[['customer_id', 'customer_num', 'customer_prefix']].head()

<>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Admin\AppData\Local\Temp\ipykernel_16544\3661262867.py:3: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df['customer_num'] = df['customer_id'].str.extract('(\d+)').astype(float)


,customer_id,customer_num,customer_prefix
0,CUST_1001,1001.0,CUST
1,CUST_1002,1002.0,CUST
2,CUST_1003,1003.0,CUST
3,CUST_1004,1004.0,CUST
4,CUST_1005,1005.0,CUST


In [140]:
# 2. Date & Time variables

df['join_date'] = pd.to_datetime(df['join_date'])

df['join_year'] = df['join_date'].dt.year
df['join_month'] = df['join_date'].dt.month
df['join_day'] = df['join_date'].dt.day
df['join_weekday'] = df['join_date'].dt.weekday

# Display extracted date features
df[['join_date', 'join_year', 'join_month', 'join_day', 'join_weekday']].head()

,join_date,join_year,join_month,join_day,join_weekday
0,2022-07-08,2022,7,8,4
1,2018-10-19,2018,10,19,4
2,2020-08-26,2020,8,26,2
3,2022-01-27,2022,1,27,3
4,2023-01-17,2023,1,17,1


**9. Encoding categorical variables (`OrdinalEncoding`, `LabelEncoding`, `OneHotEncoding`)**

In [141]:
# 1. Ordinal Encoding (education levels)

# Updated categories list including 'Primary'
edu_order = [['Primary', 'Secondary', 'Graduate', 'Post-Graduate']]
encoder = OrdinalEncoder(categories=edu_order, handle_unknown='use_encoded_value', unknown_value=-1)

df['education_encoded'] = encoder.fit_transform(df[['education_level']]).ravel()

# Display encoded results
df[['education_level', 'education_encoded']].head()

,education_level,education_encoded
0,Post-Graduate,3.0
1,Post-Graduate,3.0
2,Secondary,1.0
3,Secondary,1.0
4,Graduate,2.0


In [142]:
# 2. Label Encoding (binary features)

# Label Encoding for binary column 'gender'
le = LabelEncoder()
df['gender_encoded'] = le.fit_transform(df['gender'])

# Display encoded results
df[['gender', 'gender_encoded']].head()

,gender,gender_encoded
0,Male,1
1,Male,1
2,Female,0
3,Male,1
4,Female,0


In [143]:
# 3. One-Hot Encoding (regions, loan purpose)

# One-Hot Encoding for nominal columns 'region' and 'loan_purpose'
df_ohe = pd.get_dummies(df, columns=['region', 'loan_purpose'], drop_first=True)

# Display dummy features
df_ohe.filter(like='region_').head()

,region_North,region_South,region_West
0,False,False,False
1,False,True,False
2,False,False,True
3,False,True,False
4,True,False,False


**10. Encoding numerical features (`Binning`, `Binarization`, `Quantile Binning`, `K-Means Binning`)**

In [144]:
# 1. Binning (discretize income into groups)

# Equal-width binning for 'annual_income'
df['income_group'] = pd.cut(df['annual_income'], bins=3, labels=['Low', 'Medium', 'High'])

# Display binned results
df[['annual_income', 'income_group']].head()

,annual_income,income_group
0,7.921935e+05,Low
1,9.881499e+05,Low
2,3.904558e+05,Low
3,1.173082e+06,Low
4,1.382993e+06,Low


In [145]:
# 2. Binarization (flag if > threshold)

# Binarize 'credit_score' with threshold 700
binarizer = Binarizer(threshold=700)
df['high_credit_flag'] = binarizer.fit_transform(df[['credit_score']]).ravel()

# Display binarized results
df[['credit_score', 'high_credit_flag']].head()

,credit_score,high_credit_flag
0,708.392919,1.0
1,675.227349,0.0
2,689.492285,0.0
3,502.353629,0.0
4,618.334750,0.0


In [146]:
# 3. Quantile Binning

# Equal-frequency binning using quantiles
df['income_quantile'] = pd.qcut(df['annual_income'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

# Display quantile results
df[['annual_income', 'income_quantile']].head()

,annual_income,income_quantile
0,7.921935e+05,Q3
1,9.881499e+05,Q4
2,3.904558e+05,Q2
3,1.173082e+06,Q4
4,1.382993e+06,Q4


In [147]:
# 4. K-Means Binning

# Discretize 'annual_income' using K-Means clustering
kmeans_binner = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='kmeans')
df['income_kmeans'] = kmeans_binner.fit_transform(df[['annual_income']].fillna(df[['annual_income']].mean())).ravel()

# Display K-Means binned results
df[['annual_income', 'income_kmeans']].head()

,annual_income,income_kmeans
0,7.921935e+05,0.0
1,9.881499e+05,0.0
2,3.904558e+05,0.0
3,1.173082e+06,0.0
4,1.382993e+06,0.0


**💡 Why We Did This:**  
We completed Part E to extract numerical/temporal values from mixed strings and dates, map categorical variables into model-readable numeric formats, and group continuous numerical features into discrete bins.

<h3 style="color: #db2777; font-family: sans-serif; border-bottom: 2px solid #f472b6; padding-bottom: 5px;">
  📐 <span style="color: #be185d;">Part F:</span> Feature Scaling
</h3>

**11. Apply multiple scaling methods (`Standardization`, `Normalization`, `Min-Max Scaling`, `MaxAbs Scaling`, `Robust Scaling`)**

In [148]:
# 1. Standardization (Z-score scaling)

# Standardization (Mean=0, Std=1)
scaler_std = StandardScaler()
df['income_std'] = scaler_std.fit_transform(df[['annual_income']].fillna(df['annual_income'].mean())).ravel()

# Display scaled results
df[['annual_income', 'income_std']].head()

,annual_income,income_std
0,7.921935e+05,0.000159
1,9.881499e+05,0.226986
2,3.904558e+05,-0.464867
3,1.173082e+06,0.441052
4,1.382993e+06,0.684031


In [149]:
# 2. Normalization

# Normalization (L2 norm scaling across samples)
normalizer = Normalizer()
df['income_norm'] = normalizer.fit_transform(df[['annual_income']].fillna(df['annual_income'].mean())).ravel()

# Display scaled results
df[['annual_income', 'income_norm']].head()

,annual_income,income_norm
0,7.921935e+05,1.0
1,9.881499e+05,1.0
2,3.904558e+05,1.0
3,1.173082e+06,1.0
4,1.382993e+06,1.0


In [150]:
# 3. Min-Max Scaling

# Min-Max Scaling (Scale values between 0 and 1)
scaler_minmax = MinMaxScaler()
df['income_minmax'] = scaler_minmax.fit_transform(df[['annual_income']].fillna(df['annual_income'].mean())).ravel()

# Display scaled results
df[['annual_income', 'income_minmax']].head()

,annual_income,income_minmax
0,7.921935e+05,0.052165
1,9.881499e+05,0.069449
2,3.904558e+05,0.016730
3,1.173082e+06,0.085761
4,1.382993e+06,0.104276


In [151]:
# 4. MaxAbs Scaling 

# MaxAbs Scaling (Scale by maximum absolute value)
scaler_maxabs = MaxAbsScaler()
df['income_maxabs'] = scaler_maxabs.fit_transform(df[['annual_income']].fillna(df['annual_income'].mean())).ravel()

# Display scaled results
df[['annual_income', 'income_maxabs']].head()

,annual_income,income_maxabs
0,7.921935e+05,0.068659
1,9.881499e+05,0.085642
2,3.904558e+05,0.033841
3,1.173082e+06,0.101670
4,1.382993e+06,0.119863


In [152]:
# 5. Robust Scaling 

# Robust Scaling (Robust to outliers using IQR)
scaler_robust = RobustScaler()
df['income_robust'] = scaler_robust.fit_transform(df[['annual_income']].fillna(df['annual_income'].mean())).ravel()

# Display scaled results
df[['annual_income', 'income_robust']].head()

,annual_income,income_robust
0,7.921935e+05,0.351552
1,9.881499e+05,0.742816
2,3.904558e+05,-0.450595
3,1.173082e+06,1.112070
4,1.382993e+06,1.531196


**💡 Why We Did This:**  
We completed Part F to bring numerical features onto a uniform scale using various scaling methods, preventing large-value features from dominating model training and improving convergence.

<h3 style="color: #0891b2; font-family: sans-serif; border-bottom: 2px solid #22d3ee; padding-bottom: 5px;">
  🧩 <span style="color: #0e7490;">Part G:</span> Feature Construction & Transformation
</h3>

**12. Apply transformations (`FunctionTransformer`, `PowerTransformer`, `ColumnTransformer`)**

In [153]:
# FunctionTransformer -> log transform, reciprocal, square root

# 1.1 Log Transform (np.log1p avoids log(0) error)
log_transformer = FunctionTransformer(np.log1p)
df['income_log'] = log_transformer.fit_transform(df[['annual_income']].fillna(df['annual_income'].mean())).values.ravel()

# 1.2 Reciprocal Transform (1 / x)
reciprocal_transformer = FunctionTransformer(lambda x: 1 / (x + 1e-5))
df['income_reciprocal'] = reciprocal_transformer.fit_transform(df[['annual_income']].fillna(df['annual_income'].mean())).values.ravel()

# 1.3 Square Root Transform
sqrt_transformer = FunctionTransformer(np.sqrt)
df['income_sqrt'] = sqrt_transformer.fit_transform(df[['annual_income']].fillna(df['annual_income'].mean())).values.ravel()

# 1.4 Display transformed results
df[['annual_income', 'income_log', 'income_reciprocal', 'income_sqrt']].head()

,annual_income,income_log,income_reciprocal,income_sqrt
0,7.921935e+05,13.582562,1.262318e-06,890.052515
1,9.881499e+05,13.803591,1.011992e-06,994.057304
2,3.904558e+05,12.875073,2.561110e-06,624.864603
3,1.173082e+06,13.975146,8.524550e-07,1083.089327
4,1.382993e+06,14.139761,7.230696e-07,1176.007153


In [154]:
# 2.1 Box-Cox Transformation (Safely add offset if values <= 0 exist)
income_values = df[['annual_income']].fillna(df['annual_income'].median())
min_val = income_values.min().values[0]

# Shift values if minimum value is <= 0
if min_val <= 0:
    income_positive = income_values + abs(min_val) + 1e-5
else:
    income_positive = income_values

boxcox_pt = PowerTransformer(method='box-cox')
df['income_boxcox'] = boxcox_pt.fit_transform(income_positive).ravel()

# 2.2 Yeo-Johnson Transformation (Handles zero and negative values natively)
yeojohnson_pt = PowerTransformer(method='yeo-johnson')
df['income_yeojohnson'] = yeojohnson_pt.fit_transform(income_values).ravel()

# Display power transformed results
display(df[['annual_income', 'income_boxcox', 'income_yeojohnson']].head())

,annual_income,income_boxcox,income_yeojohnson
0,7.921935e+05,0.486168,0.486168
1,9.881499e+05,0.796765,0.796765
2,3.904558e+05,-0.658919,-0.658919
3,1.173082e+06,1.024006,1.024006
4,1.382993e+06,1.231304,1.231304


In [155]:
# 3. Define preprocessor pipeline for numeric and categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['annual_income', 'loan_amount', 'credit_score']),
        ('cat', OneHotEncoder(drop='first'), ['region', 'loan_purpose'])
    ]
)

# Fit and transform dataset
df_transformed = preprocessor.fit_transform(df.fillna(df.mean(numeric_only=True)))

# Display transformed array shape
print("Transformed Feature Matrix Shape:", df_transformed.shape)

Transformed Feature Matrix Shape: (500, 10)


**13. Construct new features (`Debt-to-Income`, `Avg Monthly Transactions`, `Spending-to-Income`)**

In [156]:
# 1. Debt-to-Income Ratio = loan_amount / annual_income
df['debt_to_income_ratio'] = df['loan_amount'] / (df['annual_income'].fillna(df['annual_income'].median()) + 1e-5)

df[['loan_amount', 'annual_income', 'debt_to_income_ratio']].head()

,loan_amount,annual_income,debt_to_income_ratio
0,126423.986511,7.921935e+05,0.159587
1,50026.940415,9.881499e+05,0.050627
2,193131.382561,3.904558e+05,0.494631
3,59596.998472,1.173082e+06,0.050804
4,114639.621982,1.382993e+06,0.082892


In [157]:
# 2. Average monthly transactions assuming 12 months in a year
df['avg_monthly_transactions'] = df['transaction_count'] / 12

df[['transaction_count', 'avg_monthly_transactions']].head()

,transaction_count,avg_monthly_transactions
0,58,4.833333
1,73,6.083333
2,44,3.666667
3,81,6.750000
4,111,9.250000


In [158]:
# 3. Spending-to-Income Ratio
df['spending_to_income_ratio'] = df['spending_ratio'] / (df['annual_income'].fillna(df['annual_income'].median()) + 1e-5)

df[['spending_ratio', 'annual_income', 'spending_to_income_ratio']].head()

,spending_ratio,annual_income,spending_to_income_ratio
0,67.459362,7.921935e+05,0.000085
1,69.981043,9.881499e+05,0.000071
2,51.584542,3.904558e+05,0.000132
3,42.549677,1.173082e+06,0.000036
4,28.876986,1.382993e+06,0.000021


**💡 Why We Did This:**  
We completed Part G to stabilize variance and normalize feature distributions using mathematical transformations (Log, Power, ColumnTransformer) and engineer domain-specific credit risk metrics (Debt-to-Income, Spending-to-Income) to increase model predictive power.

<h3 style="color: #16a34a; font-family: sans-serif; border-bottom: 2px solid #4ade80; padding-bottom: 5px;">
  📑 <span style="color: #15803d;">Part H:</span> Final Deliverable
</h3>

**14. Provide a final cleaned and transformed dataset (`Export CSV`, `Verify Shape`)**

In [159]:
# Export final processed dataset to CSV
final_df = df.copy()

# Save final dataset
final_df.to_csv('final_customer_credit_risk_cleaned.csv', index=False)

# Verify final dataset shape and preview
print("Final Cleaned Dataset Saved Successfully!")
print("Final Dataset Shape:", final_df.shape)
final_df.head()

Final Cleaned Dataset Saved Successfully!
Final Dataset Shape: (500, 47)


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,...,income_maxabs,income_robust,income_log,income_reciprocal,income_sqrt,income_boxcox,income_yeojohnson,debt_to_income_ratio,avg_monthly_transactions,spending_to_income_ratio
0,CUST_1001,58.0,Male,East,Post-Graduate,Salaried,7.921935e+05,126423.986511,Business,708.392919,...,0.068659,0.351552,13.582562,1.262318e-06,890.052515,0.486168,0.486168,0.159587,4.833333,0.000085
1,CUST_1002,48.0,Male,South,Post-Graduate,Salaried,9.881499e+05,50026.940415,Business,675.227349,...,0.085642,0.742816,13.803591,1.011992e-06,994.057304,0.796765,0.796765,0.050627,6.083333,0.000071
2,CUST_1003,34.0,Female,West,Secondary,Salaried,3.904558e+05,193131.382561,Other,689.492285,...,0.033841,-0.450595,12.875073,2.561110e-06,624.864603,-0.658919,-0.658919,0.494631,3.666667,0.000132
3,CUST_1004,62.0,Male,South,Secondary,Salaried,1.173082e+06,59596.998472,Education,502.353629,...,0.101670,1.112070,13.975146,8.524550e-07,1083.089327,1.024006,1.024006,0.050804,6.750000,0.000036
4,CUST_1005,27.0,Female,North,Graduate,Self-Employed,1.382993e+06,114639.621982,Home,618.334750,...,0.119863,1.531196,14.139761,7.230696e-07,1176.007153,1.231304,1.231304,0.082892,9.250000,0.000021


**15. Write a report summarizing (`Missing Values`, `Outliers`, `Encoding`, `Scaling`, `New Features`, `ML Readiness`)**

# 📌 Part H: Final Pipeline Summary Report

### 1. Missing Value Strategies & Effectiveness
* **SimpleImputer (Mean/Most Frequent):** Filled basic numerical (`age`) and categorical missing values without altering overall distribution.
* **KNN & MICE Imputers:** Effectively reconstructed correlated missing features (`annual_income`, `credit_score`) using multivariate relationships.

### 2. Outlier Handling Results
* **IQR & Z-score Methods:** Successfully detected extreme skewness in financial features.
* **Winsorization & Capping:** Capped extreme values at 1st and 99th percentiles to preserve sample size while eliminating influential outliers.

### 3. Encoding Methods Applied
* **Ordinal Encoding:** Applied to `education_level` preserving hierarchical order (`Primary` < `Secondary` < `Graduate` < `Post-Graduate`).
* **Label Encoding:** Applied to binary targets like `gender` and `default_flag`.
* **One-Hot Encoding:** Converted nominal features (`region`, `loan_purpose`) into binary indicator columns (`drop_first=True`).

### 4. Scaling & Transformations Applied
* **Standardization & Min-Max Scaling:** Brought continuous features (`annual_income`, `loan_amount`) to uniform scales for ML algorithms.
* **Log & Power Transformations:** Applied `np.log1p` and `PowerTransformer` (Box-Cox / Yeo-Johnson) to stabilize variance and normalize skewed distributions.

### 5. Newly Engineered Features
* **Debt-to-Income Ratio:** `loan_amount / annual_income` — direct indicator of customer repayment capacity.
* **Spending-to-Income Ratio:** Measures discretionary spending impact on credit risk.
* **Temporal Features:** Extracted `join_year`, `join_month`, and `join_weekday` from `join_date` for time-series behavior patterns.

### 6. Final Dataset Shape & Readiness
* **Dataset Readiness:** All missing values resolved, categorical data numericized, features scaled, and high-value indicators constructed.
* **Status:** Ready for Machine Learning model training (Logistic Regression, Decision Trees, Random Forest, XGBoost).

<h3 style="color: #ea580c; font-family: sans-serif; border-bottom: 2px solid #fb923c; padding-bottom: 5px;">
  📌 <span style="color: #c2410c;">Expected Outcome</span>
</h3>

#### **By the end of this mega project, we have achieved:**

* **Complete Workflow:** Planned and executed an end-to-end data preprocessing pipeline.
* **Data Cleaning:** Successfully handled missing values using MICE & KNN imputation and capped outliers.
* **Advanced Prep:** Applied ordinal/one-hot encoding and standardized continuous features.
* **Feature Engineering:** Constructed high-value metrics like Debt-to-Income and Spending-to-Income ratios.
* **ML Readiness:** Generated a clean, fully numeric `final_customer_credit_risk_cleaned.csv` dataset ready for machine learning model training.